# Task 1 Statistics Evidence - dabi0142


## Setup


In [ ]:
from importlib import import_module
import os

import pandas as pd

from data2001.common.paths import PROJECT_ROOT, resolve_project_path
from data2001.config import load_settings
from data2001.task1_cleaning.workflow import run_task1_cleaning


os.chdir(PROJECT_ROOT)

MEMBER_UNIKEY = "dabi0142"
settings = load_settings("configs/local.yaml")
statistics_module = import_module(f"data2001.task1_statistics.{MEMBER_UNIKEY}_statistics")

member_context = pd.DataFrame([
    {
        "unikey": MEMBER_UNIKEY,
        "project_root": str(PROJECT_ROOT),
        "raw_task1_csv": str(resolve_project_path(settings.outputs.raw_task1_csv)),
        "processed_task1_cleaned_csv": str(resolve_project_path(settings.outputs.processed_task1_cleaned_csv)),
    }
])
display(member_context)

## Shared Cleaning Input


In [ ]:
raw_task1_csv = resolve_project_path(settings.outputs.raw_task1_csv)
processed_task1_cleaned_csv = resolve_project_path(settings.outputs.processed_task1_cleaned_csv)

cleaned_df = run_task1_cleaning(
    str(raw_task1_csv),
    str(processed_task1_cleaned_csv),
)

display(cleaned_df.head())
display(pd.DataFrame([{"rows": len(cleaned_df), "columns": len(cleaned_df.columns)}]))

## Individual Derived Statistics


In [ ]:
results = []
errors = []

for statistic_function in statistics_module.STATISTICS:
    try:
        result = statistic_function(cleaned_df)
    except NotImplementedError:
        continue
    except Exception as exc:
        errors.append({"function": statistic_function.__name__, "error": f"{type(exc).__name__}: {exc}"})
        continue
    results.append(result.to_dict())

statistics_df = (
    pd.DataFrame(results)
    .sort_values("statistic_id")
    .reset_index(drop=True)
)

pd.set_option("display.max_colwidth", 100)
display(
    statistics_df[
        ["statistic_id", "title", "value", "unit", "description"]
    ]
)

if errors:
    display(pd.DataFrame(errors))

## Explanation Notes
